# 01 - How a detector is scored: IoU, precision, recall, AP and mAP

In notebook 00 we *looked* at detections and guessed that more boxes meant better. This notebook builds the real scoring system **from scratch** with plain NumPy, so that later numbers like `mAP50-95 = 0.31` mean something to you.

No GPU or internet is needed. It runs on Kaggle, Colab or your laptop.

| Part | Question it answers |
|---|---|
| A | How much do two boxes overlap? (IoU) |
| B | Is a predicted box correct? (true/false positives) |
| C | How good is the model at each confidence level? (precision and recall) |
| D | How do we turn that into one number? (AP) |
| E | Why do papers report mAP@0.5 and mAP@0.5:0.95? |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

## Part A - IoU (intersection over union)

A box is written `(x1, y1, x2, y2)`: top-left and bottom-right corners.

**IoU = area of overlap / area covered by either box.** It is 0 when the boxes do not touch and 1 when they are identical.

To find the overlap rectangle, take the *larger* of the two left edges, the *larger* of the two top edges, the *smaller* of the two right edges and the *smaller* of the two bottom edges. If that rectangle has negative width or height the boxes do not overlap, so we clamp it at 0.

In [ ]:
def iou(box_a, box_b):
    """IoU of two boxes given as (x1, y1, x2, y2)."""
    # corners of the overlap rectangle
    ix1 = max(box_a[0], box_b[0])
    iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2])
    iy2 = min(box_a[3], box_b[3])

    # overlap size; negative means no overlap, so clamp at 0
    inter_w = max(0.0, ix2 - ix1)
    inter_h = max(0.0, iy2 - iy1)
    inter = inter_w * inter_h

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter  # the overlap is counted twice, so subtract it once

    return inter / union if union > 0 else 0.0

Always test a metric on cases where you know the answer. Two 10x10 boxes sharing a 5x10 strip: overlap 50, union 100 + 100 - 50 = 150, so IoU = 1/3.

In [ ]:
assert iou((0, 0, 10, 10), (0, 0, 10, 10)) == 1.0           # identical boxes
assert iou((0, 0, 10, 10), (20, 20, 30, 30)) == 0.0         # far apart
assert abs(iou((0, 0, 10, 10), (5, 0, 15, 10)) - 1 / 3) < 1e-9  # the example above
print("All IoU tests passed")

In [ ]:
a, b = (0, 0, 10, 10), (5, 0, 15, 10)

fig, ax = plt.subplots(figsize=(5, 3))
for box, color in ((a, "tab:blue"), (b, "tab:orange")):
    ax.add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                           fill=False, edgecolor=color, linewidth=2))
ax.set_xlim(-2, 17)
ax.set_ylim(-2, 12)
ax.set_aspect("equal")
ax.set_title(f"IoU = {iou(a, b):.3f}")
plt.show()

## Part B - Is a prediction correct?

Vocabulary:
- **Ground truth (GT):** the real, human-labeled boxes.
- **True positive (TP):** a prediction that matches a GT box with IoU at or above a threshold (we start with 0.5).
- **False positive (FP):** a prediction that matches nothing, or that matches a GT box already claimed by a more confident prediction (a duplicate).
- **False negative (FN):** a GT box that no prediction found.

Procedure: sort predictions from most to least confident. For each one, find the GT box it overlaps most. If the IoU is high enough *and* that GT box is not already taken, it is a TP and the GT box is marked taken. Otherwise it is an FP.

Here is a small made-up example with two images and one class. There are 5 real objects. Each prediction is `(image, box, confidence)`.

In [ ]:
gts = {
    "img1": [(10, 10, 50, 50), (60, 60, 100, 100), (120, 20, 160, 60)],
    "img2": [(20, 20, 70, 70), (90, 90, 130, 130)],
}

preds = [
    ("img1", (12, 11, 52, 49), 0.95),     # good match for the first object
    ("img1", (58, 62, 98, 102), 0.90),    # good match for the second object
    ("img2", (22, 22, 72, 68), 0.85),     # good match in image 2
    ("img1", (200, 200, 240, 240), 0.80), # nothing is there: a false alarm
    ("img1", (11, 10, 51, 51), 0.70),     # a second box on an object already found: duplicate
    ("img2", (95, 100, 140, 140), 0.60),  # right object but sloppy box
]
# Note: nothing predicts the object at (120, 20, 160, 60), so it will be a false negative.

In [ ]:
def match_predictions(gts, preds, iou_thr=0.5):
    """Label every prediction TP or FP. Returns (rows, number_of_ground_truth_boxes)."""
    preds = sorted(preds, key=lambda p: -p[2])  # most confident first
    used = {img: [False] * len(boxes) for img, boxes in gts.items()}
    rows = []
    for img, box, conf in preds:
        # find the GT box in the same image that this prediction overlaps most
        best_iou, best_j = 0.0, -1
        for j, gt_box in enumerate(gts.get(img, [])):
            v = iou(box, gt_box)
            if v > best_iou:
                best_iou, best_j = v, j

        if best_iou < iou_thr:
            rows.append((img, conf, best_iou, False, "FP: overlaps too little"))
        elif used[img][best_j]:
            rows.append((img, conf, best_iou, False, "FP: duplicate of an object already found"))
        else:
            used[img][best_j] = True
            rows.append((img, conf, best_iou, True, "TP"))
    n_gt = sum(len(boxes) for boxes in gts.values())
    return rows, n_gt


rows, n_gt = match_predictions(gts, preds, iou_thr=0.5)
for img, conf, best_iou, is_tp, label in rows:
    print(f"{img}  confidence {conf:.2f}  best IoU {best_iou:.2f}  ->  {label}")
print("Ground-truth objects:", n_gt)

## Part C - Precision and recall

- **Precision = TP / (TP + FP):** of the boxes I predicted, how many were right?
- **Recall = TP / (all GT boxes):** of the real objects, how many did I find?

They pull in opposite directions. Predict almost nothing and precision is high but recall is low. Predict everything and recall is high but precision collapses.

To see the whole trade-off, walk down the list from most to least confident and recompute both after every prediction. That is exactly what the confidence threshold does: each row below is "what if I kept only predictions down to this one?"

In [ ]:
def precision_recall(rows, n_gt):
    is_tp = np.array([r[3] for r in rows], dtype=float)
    tp = np.cumsum(is_tp)        # running count of true positives
    fp = np.cumsum(1 - is_tp)    # running count of false positives
    precision = tp / (tp + fp)
    recall = tp / n_gt
    return precision, recall


precision, recall = precision_recall(rows, n_gt)
for (img, conf, *_), p, r in zip(rows, precision, recall):
    print(f"keep predictions down to confidence {conf:.2f}:  precision {p:.2f}  recall {r:.2f}")

In [ ]:
plt.figure(figsize=(5, 4))
plt.step(recall, precision, where="post")
plt.plot(recall, precision, "o")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0, 1.05)
plt.ylim(0, 1.05)
plt.title("Precision-recall curve")
plt.show()

## Part D - Average precision (AP)

One curve is hard to compare across models, so we summarize it with its **area**. A perfect detector has precision 1 at every recall level and an AP of 1.

One standard detail: before measuring the area we replace each precision value by the *highest precision at that recall or any higher recall*. This removes the little zigzags so that a single unlucky wrong box does not distort the score.

In [ ]:
def average_precision(precision, recall):
    """Area under the precision-recall curve (all-point interpolation)."""
    # add sentinel points at both ends of the curve
    r = np.concatenate(([0.0], recall, [1.0]))
    p = np.concatenate(([0.0], precision, [0.0]))

    # replace each precision by the best precision at that recall or higher (right to left)
    for i in range(len(p) - 2, -1, -1):
        p[i] = max(p[i], p[i + 1])

    # add up the rectangles wherever recall increases
    idx = np.where(r[1:] != r[:-1])[0]
    return float(np.sum((r[idx + 1] - r[idx]) * p[idx + 1]))


ap50 = average_precision(precision, recall)
print(f"AP at IoU 0.5 = {ap50:.3f}")
print("Check by hand: we found 3 of 5 objects with perfect precision (0.6 area), and never found the other 2.")

**mAP** is simply the AP computed for each class separately, then averaged over classes. This toy example has one class, so AP and mAP are the same number. VisDrone has 10 classes, so its mAP averages 10 APs.

## Part E - Why `mAP50` and `mAP50-95`?

So far a prediction counted as correct at IoU 0.5, which is a loose standard. Re-score at stricter thresholds from 0.50 to 0.95 in steps of 0.05, and average the results. That is **mAP@0.5:0.95** (also written mAP50-95). It rewards models whose boxes are *tight*, not just roughly in the right place.

In [ ]:
thresholds = np.arange(0.50, 1.00, 0.05)  # 0.50, 0.55, ..., 0.95
aps = []
for t in thresholds:
    rows_t, n_gt = match_predictions(gts, preds, iou_thr=t)
    p_t, r_t = precision_recall(rows_t, n_gt)
    aps.append(average_precision(p_t, r_t))
    print(f"IoU threshold {t:.2f}:  AP = {aps[-1]:.3f}")

print()
print(f"AP@0.5      = {aps[0]:.3f}")
print(f"AP@0.5:0.95 = {float(np.mean(aps)):.3f}")

## Your turn

1. Change the last prediction's box to `(92, 92, 132, 132)` so it matches well. What happens to AP@0.5 and to AP@0.5:0.95?
2. Add five junk predictions with confidence between 0.01 and 0.05 in empty areas. Does AP change much? Think about *where* they land in the ranked list and why.
3. In your own words: why can a model have a high mAP@0.5 but a low mAP@0.5:0.95?

Ultralytics reports the same family of numbers when it validates a model. As far as I know it uses a slightly different way of interpolating the curve, so its values will not match this notebook to every decimal place, but the idea is identical.